In [2]:
# ============================================================
# NOTEBOOK: 02_data_cleaning.ipynb
# PURPOSE:  Clean and merge all 9 Olist tables into a single
#           master_orders table ready for analysis
# DECISIONS:
#   - Filtered to 'delivered' orders only (96,470 of 99,441)
#   - Dropped rows with null delivery dates (removed ~3,000)
#   - Created actual_delivery_days and is_late features
#   - Merged English category names from translation table
#   - Aggregated payments to order level to avoid row duplication
#   - Filled 1,559 missing category names as 'uncategorized'
# OUTPUT:   data/cleaned/master_orders.csv + SQLite master_orders
# ============================================================

In [3]:
import pandas as pd
import sqlite3
import os

db_path="C:/Users/rs38129/Desktop/BI/retail-operations-analytics/Data/retail_olist.db"
connection=sqlite3.connect(db_path)

In [4]:
df_orders=pd.read_sql(""" SELECT * FROM orders WHERE order_status='delivered'""", connection)
print(f"Delivered orders : {len(df_orders)}")
df_orders.head()

Delivered orders : 96478


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [5]:
print("NULL counts in orders table:")
print(df_orders.isnull().sum())

NULL counts in orders table:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [6]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

print("Null counts in date columns:")
print(df_orders[date_columns].isnull().sum())

Null counts in date columns:
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      2
order_delivered_customer_date     8
order_estimated_delivery_date     0
dtype: int64


In [7]:
for column in date_columns:
    df_orders[column]=pd.to_datetime(df_orders[column])
print("Date columns converted to datetime format.")
print(df_orders[date_columns].dtypes)

Date columns converted to datetime format.
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [8]:
# How many days did actual delivery take?
df_orders['actual_delivery_days'] = (
    df_orders['order_delivered_customer_date'] - 
    df_orders['order_purchase_timestamp']
).dt.days

# Was the order delivered late? (1 = late, 0 = on time)
df_orders['is_late'] = (
    df_orders['order_delivered_customer_date'] > 
    df_orders['order_estimated_delivery_date']
).astype(int)

print(f"Average delivery days: {df_orders['actual_delivery_days'].mean():.1f}")
print(f"Late deliveries: {df_orders['is_late'].sum()} ({df_orders['is_late'].mean()*100:.1f}%)")

Average delivery days: 12.1
Late deliveries: 7826 (8.1%)


In [9]:
before = len(df_orders)
df_orders = df_orders.dropna(subset=[
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
after = len(df_orders)

print(f"Rows removed: {before - after}")
print(f"Clean orders remaining: {after}")

Rows removed: 8
Clean orders remaining: 96470


In [10]:
df_order_items = pd.read_sql(""" SELECT * FROM order_items """, connection)
print("Nulls in order_items:")
print(df_order_items.isnull().sum())
print(f"\nTotal items: {len(df_order_items)}")
df_order_items.head()

Nulls in order_items:
order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64

Total items: 112650


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [11]:
df_order_payments = pd.read_sql("SELECT * FROM order_payments", connection)

print("Nulls in order_payments:")
print(df_order_payments.isnull().sum())
print(f"\nPayment types:")
print(df_order_payments['payment_type'].value_counts())

Nulls in order_payments:
order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64


In [12]:
df_order_reviews = pd.read_sql("SELECT * FROM order_reviews", connection)

print("Nulls in order_reviews:")
print(df_order_reviews.isnull().sum())
print(f"\nReview score distribution:")
print(df_order_reviews['review_score'].value_counts().sort_index())

Nulls in order_reviews:
review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [13]:
products= pd.read_sql(""" SELECT * FROM products""", connection)
category_translation= pd.read_sql(""" SELECT * FROM category_translation""", connection)
products= products.merge(category_translation, on='product_category_name', how='left')
print(f"Products: {len(products)}")
print(f"Nulls in category name (English): {products['product_category_name_english'].isnull().sum()}")

Products: 32951
Nulls in category name (English): 623


In [14]:
customers = pd.read_sql(""" SELECT * FROM customers""", connection)
sellers = pd.read_sql(""" SELECT * FROM sellers""", connection)

print(f"Customers: {len(customers)}")
print(f"Sellers: {len(sellers)}")

Customers: 99441
Sellers: 3095


In [15]:
master=df_orders.copy()
master=master.merge(df_order_items, on='order_id', how='left')
master=master.merge(customers, on='customer_id', how='left')
master=master.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')
master=master.merge(sellers, on='seller_id', how='left')
# Join payments - aggregate to order level first to avoid duplicates
payments_agg = df_order_payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_type=('payment_type', 'first')
).reset_index()
master = master.merge(payments_agg, on='order_id', how='left')

# Join reviews - keep one review per order
reviews_clean = df_order_reviews.drop_duplicates(subset='order_id')[['order_id', 'review_score']]
master = master.merge(reviews_clean, on='order_id', how='left')

print(f"Master table shape: {master.shape}")
print(f"Columns: {master.columns.tolist()}")



Master table shape: (110189, 27)
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'actual_delivery_days', 'is_late', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'total_payment', 'payment_type', 'review_score']


In [16]:
print("Null counts in master table:")
print(master.isnull().sum()[master.isnull().sum() > 0])

Null counts in master table:
order_approved_at                  15
order_delivered_carrier_date        1
product_category_name_english    1559
total_payment                       3
payment_type                        3
review_score                      827
dtype: int64


In [17]:
# reviews - fill missing with median or drop
master['review_score'] = master['review_score'].fillna(master['review_score'].median())

# payments - drop 3 rows with no payment (likely data errors)
master = master.dropna(subset=['total_payment'])

In [18]:
master.to_csv("../data/cleaned/master_orders.csv", index=False)
print(f"✓ Master table exported: {len(master)} rows")

✓ Master table exported: 110186 rows


In [19]:
master['product_category_name_english']=master['product_category_name_english'].fillna('uncategorized')
print("Null check after fix:")
print(master['product_category_name_english'].isnull().sum())
print(f"\nUnique categories: {master['product_category_name_english'].nunique()}")

Null check after fix:
0

Unique categories: 72


In [20]:
master['order_year'] = master['order_purchase_timestamp'].dt.year
master['order_month'] = master['order_purchase_timestamp'].dt.month
master['order_year_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)
print("Sample of time columns:")
print(master[['order_purchase_timestamp', 'order_year', 'order_month', 'order_year_month']].head())

Sample of time columns:
  order_purchase_timestamp  order_year  order_month order_year_month
0      2017-10-02 10:56:33        2017           10          2017-10
1      2018-07-24 20:41:37        2018            7          2018-07
2      2018-08-08 08:38:49        2018            8          2018-08
3      2017-11-18 19:28:06        2017           11          2017-11
4      2018-02-13 21:18:39        2018            2          2018-02


In [21]:
master.to_csv("../data/cleaned/master_orders.csv", index=False)
print(f"✓ Final master table exported")
print(f"  Rows: {len(master)}")
print(f"  Columns: {len(master.columns)}")
print(f"\nFinal columns:")
for col in master.columns.tolist():
    print(f"  - {col}")

✓ Final master table exported
  Rows: 110186
  Columns: 30

Final columns:
  - order_id
  - customer_id
  - order_status
  - order_purchase_timestamp
  - order_approved_at
  - order_delivered_carrier_date
  - order_delivered_customer_date
  - order_estimated_delivery_date
  - actual_delivery_days
  - is_late
  - order_item_id
  - product_id
  - seller_id
  - shipping_limit_date
  - price
  - freight_value
  - customer_unique_id
  - customer_zip_code_prefix
  - customer_city
  - customer_state
  - product_category_name_english
  - seller_zip_code_prefix
  - seller_city
  - seller_state
  - total_payment
  - payment_type
  - review_score
  - order_year
  - order_month
  - order_year_month


In [22]:
connection=sqlite3.connect(db_path)
master.to_sql("master_orders", connection, if_exists="replace", index= False)
result=pd.read_sql("SELECT COUNT(*) AS total_rows FROM master_orders", connection)
print(result)
connection.close()

   total_rows
0      110186
